# DeepSDF for Peptides - Minimal Implementation

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import os, glob
from scipy.spatial import cKDTree
import matplotlib.pyplot as plt
from pathlib import Path
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import open3d as o3d
from synthetic_peptides_dataset import SyntheticPeptidesDataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


In [8]:
def compute_sdf_samples(surface_points, n_samples=4096):
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(surface_points)
    pcd.estimate_normals(search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=0.15, max_nn=50))
    pcd.orient_normals_consistent_tangent_plane(k=15)
    surface_normals = np.asarray(pcd.normals)

    tree = cKDTree(surface_points)
    near_idx = np.random.choice(len(surface_points), int(n_samples * 0.3), replace=True)
    near_points = surface_points[near_idx] + np.random.normal(0, 0.03, (len(near_idx), 3))
    bbox_min, bbox_max = surface_points.min(0) - 0.2, surface_points.max(0) + 0.2
    rand_points = np.random.uniform(bbox_min, bbox_max, (n_samples - len(near_idx), 3))
    sample_points = np.vstack([near_points, rand_points])

    distances, closest_indices = tree.query(sample_points)
    sample_vectors = sample_points - surface_points[closest_indices]
    side_indicators = np.sum(sample_vectors * surface_normals[closest_indices], axis=1)
    signed_distances = np.where((side_indicators < 0) & (distances < 0.05), -distances, distances)
    return sample_points, signed_distances


class SDFDataset(Dataset):
    def __init__(self, peptide_dataset, n_samples=4096):
        self.peptide_dataset = peptide_dataset
        self.n_samples = n_samples

    def __len__(self):
        return len(self.peptide_dataset)

    def __getitem__(self, idx):
        surface = self.peptide_dataset[idx]["points"].numpy()
        coords, sdf = compute_sdf_samples(surface, self.n_samples)
        return torch.tensor(coords, dtype=torch.float32), torch.tensor(sdf, dtype=torch.float32), idx

def compute_udf_samples(surface_points, n_samples=4096):
    tree = cKDTree(surface_points)
    near_idx = np.random.choice(len(surface_points), int(n_samples*0.3), replace=True)
    near_points = surface_points[near_idx] + np.random.normal(0, 0.03, (len(near_idx), 3))
    bbox_min, bbox_max = surface_points.min(0) - 0.2, surface_points.max(0) + 0.2
    rand_points = np.random.uniform(bbox_min, bbox_max, (n_samples - len(near_idx), 3))
    sample_points = np.vstack([near_points, rand_points])
    distances, _ = tree.query(sample_points)
    return sample_points, distances


class UDFDataset(Dataset):
    def __init__(self, peptide_dataset, n_samples=4096):
        self.peptide_dataset = peptide_dataset
        self.n_samples = n_samples

    def __len__(self):
        return len(self.peptide_dataset)

    def __getitem__(self, idx):
        surface = self.peptide_dataset[idx]["points"].numpy()
        coords, udf = compute_udf_samples(surface, self.n_samples)
        return torch.tensor(coords, dtype=torch.float32), torch.tensor(udf, dtype=torch.float32), idx


try:
    test_data = SyntheticPeptidesDataset('./data/synthetic_peptides_split/train', num_files=1, structure_types=['nanotubes'])
except:
    test_data = SyntheticPeptidesDataset('./data/synthetic_peptides', num_files=1)

surface_np = test_data[0]["points"].numpy()
sdf_coords, sample_sdf = compute_sdf_samples(surface_np, n_samples=10000)
udf_coords, sample_udf = compute_udf_samples(surface_np, n_samples=10000)

inside_mask = sample_sdf < -0.02
surface_mask = np.abs(sample_sdf) <= 0.02
outside_mask = sample_sdf > 0.02
print(f"SDF — Inside: {np.sum(inside_mask)} ({np.sum(inside_mask)/len(sample_sdf)*100:.1f}%)  "
      f"Surface: {np.sum(surface_mask)} ({np.sum(surface_mask)/len(sample_sdf)*100:.1f}%)  "
      f"Outside: {np.sum(outside_mask)} ({np.sum(outside_mask)/len(sample_sdf)*100:.1f}%)  "
      f"Range: [{sample_sdf.min():.3f}, {sample_sdf.max():.3f}]")

very_close_mask = sample_udf < 0.02
close_mask = (sample_udf >= 0.02) & (sample_udf < 0.1)
far_mask = sample_udf >= 0.1
print(f"UDF — Very close: {np.sum(very_close_mask)} ({np.sum(very_close_mask)/len(sample_udf)*100:.1f}%)  "
      f"Close: {np.sum(close_mask)} ({np.sum(close_mask)/len(sample_udf)*100:.1f}%)  "
      f"Far: {np.sum(far_mask)} ({np.sum(far_mask)/len(sample_udf)*100:.1f}%)  "
      f"Range: [{sample_udf.min():.3f}, {sample_udf.max():.3f}]")

fig = make_subplots(
    rows=1, cols=3,
    specs=[[{'type': 'scene'}, {'type': 'scene'}, {'type': 'scene'}]],
    subplot_titles=['Ground Truth Point Cloud', 'SDF Samples (Red=Outside, Blue=Inside)', 'UDF Samples']
)

# Ground truth
fig.add_trace(go.Scatter3d(
    x=surface_np[:,0], y=surface_np[:,1], z=surface_np[:,2],
    mode='markers', marker=dict(size=2, color='steelblue', opacity=0.6),
    name='Ground Truth'
), row=1, col=1)

# SDF
fig.add_trace(go.Scatter3d(
    x=surface_np[:,0], y=surface_np[:,1], z=surface_np[:,2],
    mode='markers', marker=dict(size=2, color='lightgray', opacity=0.2),
    name='Surface (SDF)', showlegend=False
), row=1, col=2)
fig.add_trace(go.Scatter3d(
    x=sdf_coords[:,0], y=sdf_coords[:,1], z=sdf_coords[:,2],
    mode='markers',
    marker=dict(size=2, color=sample_sdf, colorscale='RdBu_r', cmin=-0.2, cmax=0.2,
                colorbar=dict(title="SDF", thickness=12, len=0.6, x=0.65)),
    name='SDF Samples',
    hovertemplate='SDF: %{marker.color:.3f}<extra></extra>'
), row=1, col=2)

# UDF
fig.add_trace(go.Scatter3d(
    x=surface_np[:,0], y=surface_np[:,1], z=surface_np[:,2],
    mode='markers', marker=dict(size=2, color='lightgray', opacity=0.2),
    name='Surface (UDF)', showlegend=False
), row=1, col=3)
fig.add_trace(go.Scatter3d(
    x=udf_coords[:,0], y=udf_coords[:,1], z=udf_coords[:,2],
    mode='markers',
    marker=dict(size=2, color=sample_udf, colorscale='Viridis', cmin=0, cmax=0.3,
                colorbar=dict(title="UDF", thickness=12, len=0.6, x=1.0)),
    name='UDF Samples',
    hovertemplate='UDF: %{marker.color:.3f}<extra></extra>'
), row=1, col=3)

fig.update_layout(
    height=600, width=1500,
    scene=dict(aspectmode='data'),
    scene2=dict(aspectmode='data'),
    scene3=dict(aspectmode='data'),
)
fig.show()

Loading centroid dataset from: ./data/synthetic_peptides_split/train
Structure types: nanotubes
Selected files per structure:
  nanotubes: 1/700
Dataset ready: 1 files, target_points=None, normalize=True
SDF — Inside: 643 (6.4%)  Surface: 2090 (20.9%)  Outside: 7267 (72.7%)  Range: [-0.050, 0.662]
UDF — Very close: 2102 (21.0%)  Close: 2294 (22.9%)  Far: 5604 (56.0%)  Range: [0.001, 0.663]
